In [ ]:
from  langchain_community.document_loaders import PyPDFLoader  

In [4]:
loader=PyPDFLoader('./attention.pdf')
docs=loader.load()
docs

[Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Tr

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
final_documents=splitter.split_documents(docs)
final_documents


[Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='reproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='scholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='avaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Google Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Llion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\n

In [15]:
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [16]:
sample=embeddings.embed_documents(final_documents[0].page_content)
sample

[[-0.047706883400678635,
  0.02979893982410431,
  -0.029307572171092033,
  -0.028761694207787514,
  -0.049182645976543427,
  -0.04869541525840759,
  0.11003471165895462,
  0.02976907417178154,
  -0.006188416387885809,
  0.05534941703081131,
  0.02045215107500553,
  -0.05075651407241821,
  0.01750912331044674,
  0.008488230407238007,
  -0.04395948350429535,
  0.043411433696746826,
  -0.02037900872528553,
  -0.029790926724672318,
  0.04417199268937111,
  0.04676864296197891,
  -0.06464885175228119,
  0.07507975399494171,
  -0.011289241723716259,
  -0.004592131357640028,
  -0.015927044674754143,
  -0.0033375706989318132,
  0.011098118498921394,
  0.10217371582984924,
  0.0035181704442948103,
  -0.009191053919494152,
  0.017634963616728783,
  0.13972395658493042,
  0.05070924386382103,
  -0.027830954641103745,
  -0.003590804524719715,
  -0.017582934349775314,
  -0.01819439046084881,
  -0.005483848042786121,
  -0.0224604532122612,
  -0.04451676458120346,
  0.015791798010468483,
  -0.0529576

In [17]:
len(sample)

75

In [19]:
from langchain_chroma import Chroma

vector_db=Chroma.from_documents(embedding=embeddings,documents=final_documents)
vector_db

In [21]:
vector_db.similarity_search("rnn")

[Document(id='96c1365a-52a4-41f6-a0aa-682cde207768', metadata={'page': 8, 'source': './attention.pdf'}, page_content='constraints and is significantly longer than the input. Furthermore, RNN sequence-to-sequence'),
 Document(id='3c3dc8e7-7449-4cc4-aa3a-baff4d9d04c6', metadata={'page': 9, 'source': './attention.pdf'}, page_content='In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-'),
 Document(id='5850c54f-e625-4cb9-988e-01d409534e5b', metadata={'page': 0, 'source': './attention.pdf'}, page_content='∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and'),
 Document(id='c836a033-01e7-4aab-b688-171c460133fd', metadata={'page': 1, 'source': './attention.pdf'}, page_content='End-to-end memory networks are based on a recurrent attention mechanism instead of sequence-')]

In [33]:
import os
from dotenv import load_dotenv
load_dotenv()


groq_api_key=os.getenv('sample_qroq_api_key')

In [45]:
from langchain_groq import ChatGroq
import httpx


custom_http_client = httpx.Client(verify=False)
llm=ChatGroq(model='gemma2-9b-it',groq_api_key=groq_api_key,http_client=custom_http_client)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3958e4550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39f9495d0>, model_name='gemma2-9b-it', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x3a9189950>)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate.from_template("""
Answer the question based ONLY on the context below. If the answer isn't contained within the context,just say 'I don't know.'

Context:
{context}

Question:
{input}

Answer:
""")


# prompt=PromptTemplate.from_template("""
# Answer the question using the context below if it's relevant. 
# If the context does not help, feel free to use your own knowledge to answer the question.

# Context:
# {context}

# Question:
# {input}

# Answer:
# """)


prompt

PromptTemplate(input_variables=['context', 'input'], template="\nAnswer the question using the context below if it's relevant. \nIf the context does not help, feel free to use your own knowledge to answer the question.\n\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n")

In [130]:
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), config={'run_name': 'format_inputs'})
| PromptTemplate(input_variables=['context', 'input'], template="\nAnswer the question using the context below if it's relevant. \nIf the context does not help, feel free to use your own knowledge to answer the question.\n\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n")
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3958e4550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39f9495d0>, model_name='gemma2-9b-it', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x3a9189950>)
| StrOutputParser(), config={'run_name': 'stuff_documents_chain'})

In [131]:
# Example question to ask
question = "what is attention mechanism?"

# Step 1 & 2: Retrieve relevant documents from vectordb (usually returns docs with 'page_content')
docs = vector_db.similarity_search(question, k=4)  # top 4 relevant docs



# Step 4: Run the document chain with context and question
response = document_chain.invoke({"context": docs, "input": question})

print(response)


According to the provided text, an attention mechanism is a key component in sequence modeling and transduction.  

More specifically, it's described as a method for relating different parts of a sequence to each other.  The text highlights "self-attention" (also known as intra-attention) as a type of attention mechanism. 



In [132]:
from langchain.chains import create_retrieval_chain

retriever=vector_db.as_retriever()
retrieval_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=document_chain
)

retrieval_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x33f2624d0>), config={'run_name': 'retrieve_documents'})
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), config={'run_name': 'format_inputs'})
            | PromptTemplate(input_variables=['context', 'input'], template="\nAnswer the question using the context below if it's relevant. \nIf the context does not help, feel free to use your own knowledge to answer the question.\n\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n")
            | ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3958e4550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39f9495d0

In [133]:
question = "What is car"

answer = retrieval_chain.invoke({"input":question})  # or retrieval_chain.run(question)
print(answer)


{'input': 'What is car', 'context': [Document(id='7aa5d367-82aa-40a1-a15a-140139c27b9f', metadata={'page': 6, 'source': './attention.pdf'}, page_content='related to the syntactic'), Document(id='deaf9300-eb20-407e-9196-decc693e4016', metadata={'page': 0, 'source': './attention.pdf'}, page_content='and massively accelerating'), Document(id='0862e517-1206-4424-9aff-604805ba1d19', metadata={'page': 3, 'source': './attention.pdf'}, page_content='function of the'), Document(id='8bb21352-1750-40b6-afbc-f0149c5a66d0', metadata={'page': 8, 'source': './attention.pdf'}, page_content='constituency parsing. This task presents specific challenges: the output is subject to strong')], 'answer': 'The provided context doesn\'t contain the answer to "What is car?".  \n\nA **car** is a wheeled motor vehicle used for transportation. \n'}


In [145]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_prompt = PromptTemplate.from_template(
    "Answer the question based only on the context below:\n\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
)

context_retrieval_chain = (
    RunnableLambda(lambda x: x["question"]) |
    retriever |
    RunnableLambda(format_docs)
)

rag_chain = {
    "context": context_retrieval_chain,
    "question": RunnablePassthrough()
} | PromptTemplate(
    input_variables=["context", "question"],
    template="Answer the question based only on the context below:\n\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
) | llm | StrOutputParser()


rag_chain
# # Using the chain
# result = rag_chain.invoke({"question": "What is LangChain?"})
# print(result)

{
  context: RunnableLambda(...)
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x33f2624d0>)
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], template='Answer the question based only on the context below:\n\n{context}\n\nQuestion:\n{question}\n\nAnswer:')
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3958e4550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x39f9495d0>, model_name='gemma2-9b-it', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x3a9189950>)
| StrOutputParser()

In [146]:
response = rag_chain.invoke({"question": "What are the benefits of using Chroma for vector search?"})
print(response)


The provided text doesn't explicitly state the benefits of using Chroma for vector search.  It only mentions that a linear projection method is beneficial for implementing attention functions in vector search. 






In [153]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to hindi"),
    HumanMessage(content="Hello How are you?")
]



In [158]:
llm.invoke(messages)

AIMessage(content='नमस्ते, आप कैसे हैं? (Namaste, aap kaise hain?) \n\n\nLet me break it down:\n\n* **नमस्ते (Namaste)**:  This is a common greeting in Hindi, similar to "Hello" or "Good day".\n* **आप (aap)**: This means "you" in a formal or respectful way.\n* **कैसे हैं (kaise hain)?**: This means "how are you?".\n\n\n\n', response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 21, 'total_tokens': 121, 'completion_time': 0.181818182, 'prompt_time': 0.001326359, 'queue_time': 0.247926681, 'total_time': 0.183144541}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-6b8aa6ba-f674-4feb-8b73-018a866e1953-0', usage_metadata={'input_tokens': 21, 'output_tokens': 100, 'total_tokens': 121})

In [174]:
from langchain_core.output_parsers import StrOutputParser


parser=StrOutputParser()
StrOutputParser().invoke(llm.invoke(messages))


'नमस्ते, आप कैसे हैं? (Namaste, aap kaise hain?) \n\n\nLet me break it down:\n\n* **नमस्ते (Namaste):** This is a common greeting in Hindi, similar to "Hello" or "Hi."\n* **आप (aap):** This means "you" in a formal or respectful way.\n* **कैसे (kaise):** This means "how."\n* **हैं (hain):**  This is the verb "to be" in the present tense, used with the subject "you." \n\n\n\n'

In [175]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [176]:
prompt.invoke({"language":"French","text":"Hello"})

ChatPromptValue(messages=[SystemMessage(content='Trnaslate the following into French:'), HumanMessage(content='Hello')])

In [177]:
llm.invoke([SystemMessage(content='Trnaslate the following into French:'), HumanMessage(content='Hello')])

AIMessage(content='Bonjour \n', response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 18, 'total_tokens': 23, 'completion_time': 0.009090909, 'prompt_time': 0.001248669, 'queue_time': 0.245035541, 'total_time': 0.010339578}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-4cda5ab3-7b66-4290-8262-64396de77ecb-0', usage_metadata={'input_tokens': 18, 'output_tokens': 5, 'total_tokens': 23})

In [178]:
parser.invoke(llm.invoke([SystemMessage(content='Trnaslate the following into French:'), HumanMessage(content='Hello')]))

'Bonjour \n'

In [179]:
chain=prompt|llm|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour \n'